In [1]:
# basic lib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings('ignore')

# ml lib
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
import catboost as cat

# setting plot
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('viridis')
my_palette = 'plasma'  # Options: 'viridis', 'plasma', 'inferno', 'magma', 'cividis', 'muted'

In [2]:
# ===================================================
# SECTION 1: FEATURE ENGINEERING
# ===================================================

def create_features(df):
    """
    Creates a variety of new features specifically for the accident risk dataset.
    This function is copied from our feature_engineering.py script.
    """
    df = df.copy()
    print(f"Original shape: {df.shape}")

    # --- Interaction & Polynomial Features ---
    print("Processing interaction and polynomial features...")
    df['curvature_div_speed_limit'] = df['curvature'] / (df['speed_limit'] + 1e-6)
    df['num_lanes_div_speed_limit'] = df['num_lanes'] / (df['speed_limit'] + 1e-6)
    df['curvature_x_speed_limit'] = df['curvature'] * df['speed_limit']
    df['num_lanes_x_curvature'] = df['num_lanes'] * df['curvature']
    df['num_lanes_x_speed_limit'] = df['num_lanes'] * df['speed_limit']
    df['curvature_sq'] = df['curvature'] ** 2
    df['speed_limit_sq'] = df['speed_limit'] ** 2
    print(f"Shape after interaction features: {df.shape}")

    # --- Combined Categorical Features ---
    print("Processing combined categorical features...")
    df['lighting_weather'] = df['lighting'].astype(str) + '_' + df['weather'].astype(str)
    df['lighting_time_of_day'] = df['lighting'].astype(str) + '_' + df['time_of_day'].astype(str)
    df['road_type_lighting'] = df['road_type'].astype(str) + '_' + df['lighting'].astype(str)
    print(f"Shape after combined categorical features: {df.shape}")

    # --- Group-By Aggregation Features ---
    print("Processing group-by aggregation features...")
    cat_cols = ['road_type', 'lighting', 'weather', 'time_of_day', 'num_lanes']
    num_cols = ['curvature', 'speed_limit']
    for cat_col in cat_cols:
        for num_col in num_cols:
            df[f'{num_col}_by_{cat_col}_mean'] = df.groupby(cat_col)[num_col].transform('mean')
            df[f'{num_col}_by_{cat_col}_std'] = df.groupby(cat_col)[num_col].transform('std')
            df[f'{num_col}_by_{cat_col}_diff_mean'] = df[num_col] - df[f'{num_col}_by_{cat_col}_mean']
    print(f"Shape after aggregation features: {df.shape}")

    # --- Boolean Interaction Features ---
    print("Processing boolean interaction features...")
    df['holiday_and_school_season'] = df['holiday'] & df['school_season']
    df['signs_and_public_road'] = df['road_signs_present'] & df['public_road']
    print(f"Shape after boolean features: {df.shape}")

    return df

In [3]:
# ===================================================
# SECTION 2: DATA PREPARATION & ONE-HOT ENCODING
# ===================================================
print("Loading data...")
# It's assumed a 'test.csv' file exists in the same directory
try:
    train_df = pd.read_csv('dataset/train.csv')
    test_df = pd.read_csv('dataset/test.csv')
except FileNotFoundError:
    print("Make sure 'train.csv' and 'test.csv' are in the correct directory.")
    exit()


print("\nStarting feature engineering...")
featured_train_df = create_features(train_df)
featured_test_df = create_features(test_df)

print("\nPreparing final dataset for training...")
# Separate target variable and store test IDs
X = featured_train_df.drop(columns=['id', 'accident_risk'])
y = featured_train_df['accident_risk']
X_test = featured_test_df.drop(columns=['id'])
test_ids = featured_test_df['id']


# Identify categorical and boolean columns
categorical_cols = X.select_dtypes(include=['object']).columns
boolean_cols = X.select_dtypes(include=['bool']).columns

# Convert boolean columns to integers (0s and 1s) for both train and test
for col in boolean_cols:
    X[col] = X[col].astype(int)
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(int)

# Apply One-Hot Encoding
print(f"Applying one-hot encoding to {len(categorical_cols)} columns...")
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - crucial for consistent predictions
train_cols = X.columns
test_cols = X_test.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X[c] = 0

X_test = X_test[train_cols] # Ensure order is the same

print(f"Final training data shape: {X.shape}")
print(f"Final test data shape: {X_test.shape}")

Loading data...

Starting feature engineering...
Original shape: (517754, 14)
Processing interaction and polynomial features...
Shape after interaction features: (517754, 21)
Processing combined categorical features...
Shape after combined categorical features: (517754, 24)
Processing group-by aggregation features...
Shape after aggregation features: (517754, 54)
Processing boolean interaction features...
Shape after boolean features: (517754, 56)
Original shape: (172585, 13)
Processing interaction and polynomial features...
Shape after interaction features: (172585, 20)
Processing combined categorical features...
Shape after combined categorical features: (172585, 23)
Processing group-by aggregation features...
Shape after aggregation features: (172585, 53)
Processing boolean interaction features...
Shape after boolean features: (172585, 55)

Preparing final dataset for training...
Applying one-hot encoding to 7 columns...
Final training data shape: (517754, 79)
Final test data shape:

In [12]:
# ===================================================
# SECTION 3: MODEL TRAINING
# ===================================================
N_SPLITS = 5
kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# --- Initialize Prediction Arrays ---
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))


# --- Model Parameters (strong baselines) ---
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse', 
    'n_estimators': 3000, 
    'learning_rate': 0.02, 
    'tree_method': 'hist', 
    'device': 'cuda', 
    'random_state': 42, 
    'early_stopping_rounds': 250
}
lgb_params = {
    'objective': 'regression_l1', 
    'metric': 'rmse', 
    'n_estimators': 3000, 
    'learning_rate': 0.02, 
    'device': 'gpu', 
    'random_state': 42
}
cat_params = {
    'iterations': 3000, 
    'learning_rate': 0.02, 
    'loss_function': 'RMSE', 
    'eval_metric': 'RMSE', 
    'task_type': 'GPU', 
    'random_seed': 42, 
    'verbose': 0, 
    'early_stopping_rounds': 250
}

# --- Train XGBoost ---
print("\nTraining XGBoost...")
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model.predict(X_val)
    test_xgb += model.predict(X_test) / N_SPLITS
xgb_oof_rmse = np.sqrt(mean_squared_error(y, oof_xgb))
print(f"XGBoost OOF RMSE: {xgb_oof_rmse:.6f}")

# --- Train LightGBM ---
print("\nTraining LightGBM...")
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb[val_idx] = model.predict(X_val)
    test_lgb += model.predict(X_test) / N_SPLITS
lgb_oof_rmse = np.sqrt(mean_squared_error(y, oof_lgb))
print(f"LightGBM OOF RMSE: {lgb_oof_rmse:.6f}")

# --- Train CatBoost ---
print("\nTraining CatBoost...")
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = cat.CatBoostRegressor(**cat_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    oof_cat[val_idx] = model.predict(X_val)
    test_cat += model.predict(X_test) / N_SPLITS
cat_oof_rmse = np.sqrt(mean_squared_error(y, oof_cat))
print(f"CatBoost OOF RMSE: {cat_oof_rmse:.6f}")


Training XGBoost...
XGBoost OOF RMSE: 0.056143

Training LightGBM...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2573
[LightGBM] [Info] Number of data points in the train set: 414203, number of used features: 79
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 4GB Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 52 dense feature groups (20.54 MB) transferred to GPU in 0.023605 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 0.340000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2570
[LightGBM] [Info] Number of data points in the train set: 414203, number of used features: 79
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 4GB Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...

In [13]:
# ===================================================
# SECTION 4: ENSEMBLING
# ===================================================
print("\n" + "=" * 60)
print("Calculating weights based on model performance...")

model_scores = [xgb_oof_rmse, lgb_oof_rmse, cat_oof_rmse]
inv_scores = [1 / (score + 1e-6) for score in model_scores]
sum_inv_scores = sum(inv_scores)
weights = [score / sum_inv_scores for score in inv_scores]
w1, w2, w3 = weights[0], weights[1], weights[2]

ensemble_oof = w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cat
ensemble_rmse = np.sqrt(mean_squared_error(y, ensemble_oof))

# Also ensemble the test predictions
ensemble_test = w1 * test_xgb + w2 * test_lgb + w3 * test_cat

print(f"Best Weights -> XGB: {w1:.3f}, LGB: {w2:.3f}, CAT: {w3:.3f}")
print(f"Final Weighted Ensemble OOF RMSE: {ensemble_rmse:.6f}")
print("=" * 60)


Calculating weights based on model performance...
Best Weights -> XGB: 0.334, LGB: 0.332, CAT: 0.334
Final Weighted Ensemble OOF RMSE: 0.056126


In [9]:
# ===================================================
# SECTION 5: CREATE SUBMISSION
# ===================================================
print("\nCreating submission file...")
submission_df = pd.DataFrame({'id': test_ids, 'accident_risk': ensemble_test})

# Clip predictions to be within a valid range (e.g., 0 to 1)
submission_df['accident_risk'] = np.clip(submission_df['accident_risk'], 0, 1)

submission_df.to_csv('submission.csv', index=False)
print("✅ Submission file 'submission.csv' created successfully.")
print(submission_df.head())


Creating submission file...
✅ Submission file 'submission.csv' created successfully.
       id  accident_risk
0  517754       0.328570
1  517755       0.122663
2  517756       0.221945
3  517757       0.389955
4  517758       0.488969
